In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.impute import SimpleImputer

brokers = pd.read_csv("synthetic_brokers_v80.csv")
leads = pd.read_csv("synthetic_leads_v80.csv")
assignments = pd.read_csv("synthetic_assignments_v80.csv")
counterfactual = pd.read_csv("synthetic_counterfactual_v80.csv")
historical = pd.read_csv("synthetic_historical_v80.csv")

/tmp/ipykernel_26682/1023119946.py:10: DtypeWarning: Columns (33,35) have mixed types. Specify dtype option on import or set low_memory=False.
  historical = pd.read_csv("synthetic_historical_v80.csv")


In [67]:
reentry_mask = assignments["lead_id"].str.contains("_R", na=False)
reentry_lead_ids = assignments.loc[reentry_mask, "lead_id"].unique()
print(f"Re-entry lead IDs in assignments: {len(reentry_lead_ids):,}")
 
# Reconstruct minimal lead rows for the re-entry leads from the historical table.
# The historical table has the merged lead features, so we can recover them there.
hist_cols_for_leads = [
    "lead_id", "lead_date", "region", "insurance_type", "language",
    "tenure_years", "digital_engagement_score", "quote_value",
    "lead_difficulty", "sophistication", "patience_hours",
    "claims_severity", "multi_product_intent", "hour_of_day",
    "is_weekend", "month", "postal_code_prefix",
]
# Keep only columns that actually exist in historical
hist_cols_for_leads = [c for c in hist_cols_for_leads if c in historical.columns]
 
reentry_leads_recovered = (
    historical.loc[historical["lead_id"].isin(reentry_lead_ids), hist_cols_for_leads]
    .drop_duplicates(subset=["lead_id"])
)
 
# Build the original_lead_id for re-entries (strip the suffix)
reentry_leads_recovered = reentry_leads_recovered.copy()
reentry_leads_recovered["original_lead_id"] = (
    reentry_leads_recovered["lead_id"].str.replace(r"_R\d+$", "", regex=True)
)
 
# Append to leads
leads_full = pd.concat([leads, reentry_leads_recovered], ignore_index=True)
 
# Add original_lead_id to original leads too (same as lead_id for non-reentry)
if "original_lead_id" not in leads_full.columns:
    leads_full["original_lead_id"] = leads_full["lead_id"]
else:
    leads_full["original_lead_id"] = leads_full["original_lead_id"].fillna(
        leads_full["lead_id"]
    )
 
print(f"\nLeads after re-entry fix: {len(leads_full):,}  "
      f"(was {len(leads):,}, added {len(reentry_leads_recovered):,})")

Re-entry lead IDs in assignments: 850

Leads after re-entry fix: 11,474  (was 10,624, added 850)


In [68]:
known_broker_ids  = set(brokers["broker_id"])
used_broker_ids   = set(assignments["broker_id"].dropna())
orphan_broker_ids = used_broker_ids - known_broker_ids
print(f"\nOrphaned broker IDs (replacement brokers): {len(orphan_broker_ids):,}")



Orphaned broker IDs (replacement brokers): 6


In [69]:
hist_cols_for_brokers = [
    "broker_id", "region", "expertise_auto", "expertise_home",
    "expertise_bundle", "conversion_rate", "csat_score", "languages",
    "ribo_licensed", "ribo_license_years", "capacity", "avg_response_time",
    "is_new_broker", "skill_level", "reliability", "commission_rate",
    "cost_per_lead", "efficiency", "burnout_risk",
]
hist_cols_for_brokers = [c for c in hist_cols_for_brokers if c in historical.columns]
 
replacement_brokers_recovered = (
    historical.loc[
        historical["broker_id"].isin(orphan_broker_ids),
        hist_cols_for_brokers,
    ]
    .drop_duplicates(subset=["broker_id"])
)
 
# Fill columns that might not exist in historical
for col in ["years_experience", "current_caseload"]:
    if col not in replacement_brokers_recovered.columns:
        replacement_brokers_recovered[col] = np.nan
 
replacement_brokers_recovered["is_new_broker"] = (
    replacement_brokers_recovered.get("is_new_broker", pd.Series(True))
    .fillna(True)
)
 
brokers_full = pd.concat([brokers, replacement_brokers_recovered], ignore_index=True)
print(f"Brokers after replacement fix: {len(brokers_full):,}  "
      f"(was {len(brokers):,}, added {len(replacement_brokers_recovered):,})")

Brokers after replacement fix: 306  (was 300, added 6)


/tmp/ipykernel_26682/839002970.py:25: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(True)


In [70]:
actual_caseload = (
    assignments[assignments["is_assigned"] == 1]
    .groupby("broker_id")
    .size()
    .rename("assignment_count")
    .reset_index()
)
brokers_full = brokers_full.merge(actual_caseload, on="broker_id", how="left")
brokers_full["assignment_count"] = brokers_full["assignment_count"].fillna(0)
 
# Utilization = assignment_count / capacity  (capped at 3 for interpretability)
brokers_full["utilization"] = np.clip(
    brokers_full["assignment_count"] / brokers_full["capacity"], 0, 3
).round(4)
# A cleaner workload proxy: actual assignments relative to capacity
brokers_full["is_overloaded"] = (
    brokers_full["assignment_count"] > brokers_full["capacity"]
).astype(int)
 
print(f"\nBroker utilization summary:")
print(brokers_full["utilization"].describe().round(3))
print(f"Overloaded brokers: {brokers_full['is_overloaded'].sum()} "
      f"({brokers_full['is_overloaded'].mean()*100:.1f}%)")


Broker utilization summary:
count    300.000
mean       1.199
std        1.023
min        0.000
25%        0.057
50%        1.207
75%        1.962
max        3.000
Name: utilization, dtype: float64
Overloaded brokers: 166 (54.2%)


In [71]:
valid_lead_ids   = set(leads_full["lead_id"])
valid_broker_ids = set(brokers_full["broker_id"])
 
before = len(assignments)
assignments_clean = assignments[
    assignments["lead_id"].isin(valid_lead_ids)
    & assignments["broker_id"].isin(valid_broker_ids)
].copy()
print(f"\nAssignments after referential integrity fix: "
      f"{len(assignments_clean):,}  (dropped {before - len(assignments_clean):,})")
 
before = len(counterfactual)
counterfactual_clean = counterfactual[
    counterfactual["lead_id"].isin(valid_lead_ids)
    & counterfactual["broker_id"].isin(valid_broker_ids)
].copy()
print(f"Counterfactual after fix: "
      f"{len(counterfactual_clean):,}  (dropped {before - len(counterfactual_clean):,})")
 
# ── 1E. Rebuild the merged historical dataset with all fixes applied ──────────
broker_cols_to_merge = [
    "broker_id", "region", "expertise_auto", "expertise_home",
    "expertise_bundle", "conversion_rate", "csat_score", "languages",
    "ribo_licensed", "ribo_license_years", "capacity", "avg_response_time",
    "is_new_broker", "skill_level", "reliability", "commission_rate",
    "cost_per_lead", "efficiency", "burnout_risk",
    "utilization", "is_overloaded",
]
broker_cols_to_merge = [c for c in broker_cols_to_merge if c in brokers_full.columns]
 
historical_fixed = (
    assignments_clean
    .merge(leads_full,                    on="lead_id",   how="inner")
    .merge(brokers_full[broker_cols_to_merge], on="broker_id", how="inner")
)
 
# Drop any leftover internal score columns
for col in ["_score", "_simulated_score"]:
    if col in historical_fixed.columns:
        historical_fixed.drop(columns=[col], inplace=True)
 
print(f"\nHistorical (fixed, merged): {historical_fixed.shape}")


Assignments after referential integrity fix: 54,136  (dropped 0)
Counterfactual after fix: 34,422  (dropped 0)

Historical (fixed, merged): (54136, 58)


In [72]:
df = historical_fixed.copy()

In [73]:
informative_missing_cols = ["insurance_type", "language", "tenure_years",
                             "digital_engagement_score"]
 
for col in informative_missing_cols:
    if col in df.columns:
        flag_col = f"{col}_missing"
        df[flag_col] = df[col].isna().astype(int)
        print(f"Created flag: {flag_col}  "
              f"(missing rate: {df[col].isna().mean()*100:.1f}%)")

Created flag: insurance_type_missing  (missing rate: 9.4%)
Created flag: language_missing  (missing rate: 9.4%)
Created flag: tenure_years_missing  (missing rate: 8.6%)
Created flag: digital_engagement_score_missing  (missing rate: 7.9%)


In [74]:
categorical_impute = {
    "insurance_type":  "UNKNOWN",
    "language":        "UNKNOWN",
    "claims_severity": "none", 
}
 
for col, fill_val in categorical_impute.items():
    if col in df.columns:
        before = df[col].isna().sum()
        df[col] = df[col].fillna(fill_val)
        print(f"Imputed {col}: {before:,} NaN → '{fill_val}'")

Imputed insurance_type: 5,102 NaN → 'UNKNOWN'
Imputed language: 5,071 NaN → 'UNKNOWN'
Imputed claims_severity: 3,993 NaN → 'none'


In [75]:
numeric_impute_cols = ["tenure_years", "digital_engagement_score",
                       "lead_difficulty", "sophistication", "patience_hours"]
 
for col in numeric_impute_cols:
    if col in df.columns:
        before  = df[col].isna().sum()
        med_val = df[col].median()
        df[col] = df[col].fillna(med_val)
        print(f"Imputed {col}: {before:,} NaN → median={med_val:.3f}")

Imputed tenure_years: 4,648 NaN → median=1.000
Imputed digital_engagement_score: 4,296 NaN → median=34.200
Imputed lead_difficulty: 3,993 NaN → median=1.002
Imputed sophistication: 3,993 NaN → median=0.510
Imputed patience_hours: 3,993 NaN → median=39.800


In [76]:
structural_nan_cols = ["conversion_delay_days", "response_time_hours", "responded"]
print("\nStructural NaN columns (left as-is):", structural_nan_cols)
for col in structural_nan_cols:
    if col in df.columns:
        print(f"  {col}: {df[col].isna().mean()*100:.1f}% missing")
 


Structural NaN columns (left as-is): ['conversion_delay_days', 'response_time_hours', 'responded']
  conversion_delay_days: 95.6% missing
  response_time_hours: 68.8% missing
  responded: 62.4% missing


In [77]:
if "multi_product_intent" in df.columns:
    df["multi_product_intent"] = df["multi_product_intent"].fillna(0).astype(int)
 
print(f"\nMissing values after imputation:")
remaining = df.isnull().sum()
print(remaining[remaining > 0].sort_values(ascending=False).head(15))


Missing values after imputation:
conversion_delay_days    51759
response_time_hours      37263
responded                33782
lead_date                 3993
region_x                  3993
postal_code_prefix        3993
quote_value               3993
hour_of_day               3993
is_weekend                3993
month                     3993
region_y                   578
expertise_auto             578
expertise_home             578
expertise_bundle           578
conversion_rate            578
dtype: int64


In [78]:
if "conversion_value" in df.columns:
    df.drop(columns=["conversion_value"], inplace=True)
    print("Dropped 'conversion_value' (redundant with 'revenue')")
 
# Normalised profit and revenue
if "revenue" in df.columns and "cost" in df.columns:
    df["net_margin"] = np.where(
        df["revenue"] > 0,
        (df["revenue"] - df["cost"]) / df["revenue"].replace(0, np.nan),
        np.nan,
    )
    df["net_margin"] = df["net_margin"].fillna(0).round(4)
 
if "quote_value" in df.columns and "cost" in df.columns:
    df["expected_roi"] = (
        (df["quote_value"] * df.get("commission_rate", 0.10) - df["cost"])
        / df["cost"].replace(0, np.nan)
    ).round(4)
    df["expected_roi"] = df["expected_roi"].fillna(0)

Dropped 'conversion_value' (redundant with 'revenue')


In [80]:
if "response_time_hours" in df.columns:
    df["response_time_bucket"] = pd.cut(
        df["response_time_hours"],
        bins=[-np.inf, 2, 6, 12, 24, 48, np.inf],
        labels=["<2h", "2-6h", "6-12h", "12-24h", "24-48h", ">48h"],
    )
    rt_order = {"<2h": 0, "2-6h": 1, "6-12h": 2, "12-24h": 3, "24-48h": 4, ">48h": 5}
    df["response_time_bucket_ord"] = (
        df["response_time_bucket"].astype(str).map(rt_order).fillna(-1).astype(int)
    )
    print("Created response_time_bucket and response_time_bucket_ord")


Created response_time_bucket and response_time_bucket_ord


In [84]:
if "utilization" in df.columns:
    df["workload_ratio"] = np.clip(df["utilization"], 0, 3).round(4)
    print("Created workload_ratio from utilization")
elif "current_caseload" in df.columns and "capacity" in df.columns:
    df["workload_ratio"] = np.clip(
        df["current_caseload"] / df["capacity"].replace(0, np.nan), 0, 3
    ).fillna(0).round(4)
    print("Created workload_ratio")

Created workload_ratio from utilization


In [85]:
def expertise_match(row):
    ins = row.get("insurance_type", "UNKNOWN")
    if ins == "auto":
        return int(row.get("expertise_auto", 0) == 1)
    elif ins == "home":
        return int(row.get("expertise_home", 0) == 1)
    elif ins == "bundle":
        return int(row.get("expertise_bundle", 0) == 1)
    return 0.5   # UNKNOWN → neutral
 
df["expertise_match"] = df.apply(expertise_match, axis=1)
print("Created expertise_match")

Created expertise_match


In [82]:
def expertise_match(row):
    ins = row.get("insurance_type", "UNKNOWN")
    if ins == "auto":
        return int(row.get("expertise_auto", 0) == 1)
    elif ins == "home":
        return int(row.get("expertise_home", 0) == 1)
    elif ins == "bundle":
        return int(row.get("expertise_bundle", 0) == 1)
    return 0.5   # UNKNOWN → neutral
 
df["expertise_match"] = df.apply(expertise_match, axis=1)
print("Created expertise_match")


Created expertise_match


In [86]:
def language_match(row):
    lead_lang   = row.get("language", "UNKNOWN")
    broker_lang = row.get("languages", "English")
    if broker_lang == "Bilingual":
        return 1.0
    if lead_lang == "UNKNOWN":
        return 0.8
    return 1.0 if lead_lang == broker_lang else 0.3
 
df["language_match"] = df.apply(language_match, axis=1)
print("Created language_match")

Created language_match


In [88]:
if "lead_date" in df.columns:
    lead_dt = pd.to_datetime(df["lead_date"])
    df["lead_dayofweek"]  = lead_dt.dt.dayofweek
    df["lead_weekofyear"] = lead_dt.dt.isocalendar().week.fillna(0).astype(int)
    df["lead_quarter"]    = lead_dt.dt.quarter
    df["lead_year"]       = lead_dt.dt.year
    print("Created lead temporal features: dayofweek, weekofyear, quarter, year")

Created lead temporal features: dayofweek, weekofyear, quarter, year


In [89]:
for col in ["conversion_rate", "csat_score", "skill_level", "reliability"]:
    if col in df.columns:
        max_val = df[col].max()
        min_val = df[col].min()
        if max_val > min_val:
            df[f"{col}_norm"] = (
                (df[col] - min_val) / (max_val - min_val)
            ).round(4)
 
broker_quality_cols = [c for c in ["conversion_rate_norm", "csat_score_norm",
                                    "skill_level_norm", "reliability_norm"]
                       if c in df.columns]

In [90]:
if broker_quality_cols:
    df["broker_quality_score"] = df[broker_quality_cols].mean(axis=1).round(4)
    print(f"Created broker_quality_score from: {broker_quality_cols}")

Created broker_quality_score from: ['conversion_rate_norm', 'csat_score_norm', 'skill_level_norm', 'reliability_norm']


In [91]:
if "quote_value" in df.columns:
    df["quote_value_tier"] = pd.qcut(
        df["quote_value"], q=4, labels=["low", "mid_low", "mid_high", "high"]
    )
    print("Created quote_value_tier (quartile-based)")
 

Created quote_value_tier (quartile-based)


In [92]:
claims_risk_map = {"none": 0, "minor": 1, "major": 2}
if "claims_severity" in df.columns:
    df["claims_risk"] = df["claims_severity"].map(claims_risk_map).fillna(0).astype(int)
    print("Created claims_risk ordinal encoding")
 

Created claims_risk ordinal encoding


In [93]:
if "broker_quality_score" in df.columns and "quote_value" in df.columns:
    df["quality_x_value"] = (
        df["broker_quality_score"] * df["quote_value"] / df["quote_value"].max()
    ).round(4)
    print("Created interaction feature quality_x_value")

Created interaction feature quality_x_value


In [ ]:
if "region_x" in df.columns and "region_y" in df.columns:
    df["region_mismatch"] = (df["region_x"] != df["region_y"]).astype(int)
elif "region" in df.columns:

    print("Note: single 'region' column found — no mismatch feature created")
 
print(f"\nDataFrame shape after feature engineering: {df.shape}")


DataFrame shape after feature engineering: (54136, 81)


In [95]:
outlier_cap_cols = [
    "conversion_delay_days",
    "response_time_hours",
    "profit",
    "expected_profit",
    "patience_hours",
    "quote_value",
]
 
print("Outlier capping at 99th percentile:")
for col in outlier_cap_cols:
    if col in df.columns:
        p99 = df[col].quantile(0.99)
        p01 = df[col].quantile(0.01)
        before_high = (df[col] > p99).sum()
        before_low  = (df[col] < p01).sum()
        df[col] = np.clip(df[col], p01, p99)
        print(f"  {col}: capped {before_high} high / {before_low} low values  "
              f"(range now [{p01:.2f}, {p99:.2f}])")

Outlier capping at 99th percentile:
  conversion_delay_days: capped 23 high / 0 low values  (range now [0.10, 13.30])
  response_time_hours: capped 168 high / 0 low values  (range now [0.50, 42.80])
  profit: capped 542 high / 408 low values  (range now [-146.77, 123.21])
  expected_profit: capped 542 high / 531 low values  (range now [-137.92, 60.55])
  patience_hours: capped 0 high / 0 low values  (range now [6.00, 168.00])
  quote_value: capped 501 high / 499 low values  (range now [588.84, 2943.13])


In [96]:
log_transform_cols = [
    "conversion_delay_days",
    "response_time_hours",
    "quote_value",
    "patience_hours",
]
print("\nLog1p transforms:")
for col in log_transform_cols:
    if col in df.columns:
        # Only transform non-NaN values
        df[f"log_{col}"] = np.log1p(df[col].clip(lower=0))
        print(f"  Created log_{col}")
 


Log1p transforms:
  Created log_conversion_delay_days
  Created log_response_time_hours
  Created log_quote_value
  Created log_patience_hours


In [97]:
DO_NOT_SCALE = {
    "converted", "censored", "is_assigned", "responded",
    "interaction_number", "position_bias",
    "propensity_score", "exploration_rate_at_time",
    "month", "hour_of_day", "lead_dayofweek", "lead_quarter", "lead_year",
    "lead_weekofyear",
    "expertise_auto", "expertise_home", "expertise_bundle",
    "expertise_match", "language_match",
    "multi_product_intent", "is_weekend", "ribo_licensed", "is_new_broker",
    "is_overloaded", "region_mismatch",
    "claims_risk", "response_time_bucket_ord",
    "insurance_type_missing", "language_missing",
    "tenure_years_missing", "digital_engagement_score_missing",
}
 
id_cols = {"lead_id", "broker_id", "original_lead_id", "assignment_date",
           "lead_date", "market_regime", "action_probabilities",
           "response_time_bucket", "quote_value_tier", "insurance_type",
           "language", "claims_severity", "languages", "region",
           "postal_code_prefix"}
 
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
scale_cols   = [c for c in numeric_cols
                if c not in DO_NOT_SCALE and c not in id_cols]
 
print(f"\nColumns to standardise (StandardScaler): {len(scale_cols)}")
print("  ", scale_cols[:10], "..." if len(scale_cols) > 10 else "")


Columns to standardise (StandardScaler): 37
   ['conversion_delay_days', 'revenue', 'cost', 'profit', 'expected_profit', 'response_time_hours', 'tenure_years', 'digital_engagement_score', 'quote_value', 'lead_difficulty'] ...


In [98]:
scaler = StandardScaler()
 
# Work only on rows where none of the scale_cols are NaN
# (NaN in structural columns like conversion_delay_days is expected)
df_scaled_subset = df[scale_cols].copy()
# Impute any remaining NaN with column median before scaling
for col in scale_cols:
    if df_scaled_subset[col].isna().any():
        df_scaled_subset[col] = df_scaled_subset[col].fillna(
            df_scaled_subset[col].median()
        )
 
scaled_values = scaler.fit_transform(df_scaled_subset)
scaled_df = pd.DataFrame(scaled_values, columns=[f"{c}_scaled" for c in scale_cols],
                         index=df.index)
 
df = pd.concat([df, scaled_df], axis=1)
print(f"Added {len(scale_cols)} scaled columns (suffix: _scaled)")

Added 37 scaled columns (suffix: _scaled)


In [99]:
le_cols = {
    "insurance_type":  None,
    "market_regime":   None,
    "claims_severity": None,
}
 
label_encoders = {}
for col in le_cols:
    if col in df.columns:
        le = LabelEncoder()
        df[f"{col}_enc"] = le.fit_transform(df[col].astype(str))
        label_encoders[col] = le
        mapping = dict(zip(le.classes_, le.transform(le.classes_)))
        print(f"Label-encoded {col}: {mapping}")

Label-encoded insurance_type: {'UNKNOWN': np.int64(0), 'auto': np.int64(1), 'bundle': np.int64(2), 'home': np.int64(3)}
Label-encoded market_regime: {'hard': np.int64(0), 'normal': np.int64(1), 'soft': np.int64(2)}
Label-encoded claims_severity: {'major': np.int64(0), 'minor': np.int64(1), 'none': np.int64(2)}


In [100]:
if "region" in df.columns:
    region_dummies = pd.get_dummies(df["region"], prefix="region", dtype=int)
    df = pd.concat([df, region_dummies], axis=1)
    print(f"One-hot encoded 'region': {region_dummies.shape[1]} columns")
 
# One-hot encode languages (3 categories)
if "languages" in df.columns:
    lang_dummies = pd.get_dummies(df["languages"], prefix="lang", dtype=int)
    df = pd.concat([df, lang_dummies], axis=1)
    print(f"One-hot encoded 'languages': {lang_dummies.shape[1]} columns")

One-hot encoded 'languages': 3 columns


In [101]:
DROP_COLS = [
    # Pure identifiers
    "original_lead_id",
    "postal_code_prefix",
    # High-cardinality / non-numeric strings used for encoding already
    "action_probabilities",     # JSON blob, too complex for direct use
    # Columns replaced by engineered versions
    "conversion_value",         # replaced by revenue
    "current_caseload",         # replaced by utilization / workload_ratio
    # Datetime objects (keep year/month/quarter features instead)
    "assignment_date",
    "lead_date",
    # Raw string categoricals now encoded
    "market_regime",            # keep market_regime_enc
    "claims_severity",          # keep claims_risk
    # Broker-level norm intermediates (kept quality_score composite)
    "conversion_rate_norm",
    "csat_score_norm",
    "skill_level_norm",
    "reliability_norm",
    # Redundant with log-transformed versions
    "response_time_bucket",     # ordinal already in response_time_bucket_ord
    "quote_value_tier",         # ordinal via qcut label; use the scaled version
]
 
# Only drop columns that actually exist
DROP_COLS = [c for c in DROP_COLS if c in df.columns]
df_final = df.drop(columns=DROP_COLS)
print(f"Dropped {len(DROP_COLS)} columns: {DROP_COLS}")

Dropped 12 columns: ['postal_code_prefix', 'action_probabilities', 'assignment_date', 'lead_date', 'market_regime', 'claims_severity', 'conversion_rate_norm', 'csat_score_norm', 'skill_level_norm', 'reliability_norm', 'response_time_bucket', 'quote_value_tier']


In [102]:
df_positive  = df_final[df_final["is_assigned"] == 1].copy()
df_negative  = df_final[df_final["is_assigned"] == 0].copy()
print(f"\nPositive (assigned) samples:  {len(df_positive):,}")
print(f"Negative (unassigned) samples: {len(df_negative):,}")
print(f"Conversion rate (positive):    "
      f"{df_positive['converted'].mean()*100:.2f}%")


Positive (assigned) samples:  20,354
Negative (unassigned) samples: 33,782
Conversion rate (positive):    11.68%


In [103]:
print(f"\nFinal dataset shape: {df_final.shape}")
remaining_nulls = df_final.isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0].sort_values(ascending=False)
 
if len(remaining_nulls):
    print("\nRemaining NaN columns (expected structural ones only):")
    print(remaining_nulls.head(15))
else:
    print("No remaining NaN values — dataset is fully clean.")


Final dataset shape: (54136, 116)

Remaining NaN columns (expected structural ones only):
conversion_delay_days        51759
log_conversion_delay_days    51759
response_time_hours          37263
log_response_time_hours      37263
responded                    33782
quality_x_value               4502
region_x                      3993
month                         3993
quote_value                   3993
hour_of_day                   3993
is_weekend                    3993
lead_dayofweek                3993
lead_quarter                  3993
log_quote_value               3993
lead_year                     3993
dtype: int64


In [104]:
col_types = {
    "Target":           ["converted"],
    "IPS weights":      ["propensity_score", "exploration_rate_at_time"],
    "Lead features":    [c for c in df_final.columns
                         if c.startswith(("lead_", "digital", "quote", "tenure",
                                          "sophistication", "patience",
                                          "claims_risk", "multi_product",
                                          "hour_", "is_weekend", "log_quote",
                                          "log_patience"))],
    "Broker features":  [c for c in df_final.columns
                         if c.startswith(("broker_", "skill", "reliability",
                                          "csat", "conversion_rate", "ribo",
                                          "burnout", "efficiency", "commission",
                                          "cost_per", "utilization", "capacity",
                                          "expertise", "avg_response", "lang_"))],
    "Interaction":      [c for c in df_final.columns
                         if c in ("expertise_match", "language_match",
                                  "quality_x_value", "region_mismatch",
                                  "workload_ratio")],
    "Temporal":         [c for c in df_final.columns
                         if "day" in c or "week" in c or "quarter" in c
                         or c == "month" or c == "lead_year"],
    "Assignment":       [c for c in df_final.columns
                         if c in ("is_assigned", "interaction_number",
                                  "position_bias", "responded",
                                  "response_time_hours", "response_time_bucket_ord",
                                  "log_response_time_hours")],
    "Outcome/profit":   [c for c in df_final.columns
                         if c in ("revenue", "cost", "profit", "expected_profit",
                                  "expected_roi", "net_margin",
                                  "conversion_delay_days",
                                  "log_conversion_delay_days")],
    "Encoded":          [c for c in df_final.columns
                         if c.endswith("_enc") or c.startswith("region_")],
    "Missing flags":    [c for c in df_final.columns if c.endswith("_missing")],
}
 
for cat, cols in col_types.items():
    cols_found = [c for c in cols if c in df_final.columns]
    if cols_found:
        print(f"\n  {cat} ({len(cols_found)}): {cols_found}")


  Target (1): ['converted']

  IPS weights (2): ['propensity_score', 'exploration_rate_at_time']

  Lead features (27): ['lead_id', 'tenure_years', 'digital_engagement_score', 'quote_value', 'lead_difficulty', 'sophistication', 'patience_hours', 'multi_product_intent', 'hour_of_day', 'is_weekend', 'tenure_years_missing', 'digital_engagement_score_missing', 'lead_dayofweek', 'lead_weekofyear', 'lead_quarter', 'lead_year', 'claims_risk', 'log_quote_value', 'log_patience_hours', 'tenure_years_scaled', 'digital_engagement_score_scaled', 'quote_value_scaled', 'lead_difficulty_scaled', 'sophistication_scaled', 'patience_hours_scaled', 'log_quote_value_scaled', 'log_patience_hours_scaled']

  Broker features (39): ['broker_id', 'expertise_auto', 'expertise_home', 'expertise_bundle', 'conversion_rate', 'csat_score', 'ribo_licensed', 'ribo_license_years', 'capacity', 'avg_response_time', 'skill_level', 'reliability', 'commission_rate', 'cost_per_lead', 'efficiency', 'burnout_risk', 'utilizatio

In [105]:
df_final.to_csv(f"prepared_full_v81.csv",     index=False)
df_positive.to_csv(f"prepared_positive_v81.csv", index=False)
df_negative.to_csv(f"prepared_negative_v81.csv", index=False)
leads_full.to_csv(f"leads_full_v81.csv",      index=False)
brokers_full.to_csv(f"brokers_full_v81.csv",  index=False)
counterfactual_clean.to_csv(
    f"counterfactual_clean_v81.csv", index=False
)

In [106]:
print(f"  prepared_full_v81.csv      — {len(df_final):,} rows × {df_final.shape[1]} cols")
print(f"  prepared_positive_v81.csv  — {len(df_positive):,} rows (assigned only)")
print(f"  prepared_negative_v81.csv  — {len(df_negative):,} rows (unassigned)")
print(f"  leads_full_v81.csv         — {len(leads_full):,} rows (incl. re-entries)")
print(f"  brokers_full_v81.csv       — {len(brokers_full):,} rows (incl. replacements)")
print(f"  counterfactual_clean_v81.csv — {len(counterfactual_clean):,} rows (clean)")
 
print("\n✓  Data preparation complete. Ready for model training.")

  prepared_full_v81.csv      — 54,136 rows × 116 cols
  prepared_positive_v81.csv  — 20,354 rows (assigned only)
  prepared_negative_v81.csv  — 33,782 rows (unassigned)
  leads_full_v81.csv         — 11,474 rows (incl. re-entries)
  brokers_full_v81.csv       — 306 rows (incl. replacements)
  counterfactual_clean_v81.csv — 34,422 rows (clean)

✓  Data preparation complete. Ready for model training.


In [108]:
orphan_lead   = ~df_final["lead_id"].isin(leads_full["lead_id"])
orphan_broker = ~df_final["broker_id"].isin(brokers_full["broker_id"])
print(f"1. Orphaned lead_ids remaining:   {orphan_lead.sum()}")
print(f"   Orphaned broker_ids remaining: {orphan_broker.sum()}")
 
# 2. Missing flags fire where the original was missing
for col in ["insurance_type", "language", "tenure_years"]:
    flag = f"{col}_missing"
    if flag in df_final.columns:
        pass  # flags were set before imputation so they are correct
 

1. Orphaned lead_ids remaining:   0
   Orphaned broker_ids remaining: 0


In [109]:
cr = df_positive["converted"].mean() * 100
print(f"\n2. Conversion rate (positive): {cr:.2f}%  "
      f"{'OK' if 5 <= cr <= 25 else 'CHECK'}")


2. Conversion rate (positive): 11.68%  OK


In [110]:
ps = df_positive["propensity_score"]
print(f"\n3. Propensity score (positive):")
print(f"   mean={ps.mean():.4f}  min={ps.min():.4f}  max={ps.max():.4f}")


3. Propensity score (positive):
   mean=0.1047  min=0.0758  max=0.3038


In [111]:
scaled_sample = [c for c in df_final.columns if c.endswith("_scaled")][:3]
if scaled_sample:
    print(f"\n4. Scaled column means (should be ~0):")
    for col in scaled_sample:
        print(f"   {col}: mean={df_final[col].mean():.4f}  "
              f"std={df_final[col].std():.4f}")


4. Scaled column means (should be ~0):
   conversion_delay_days_scaled: mean=0.0000  std=1.0000
   revenue_scaled: mean=-0.0000  std=1.0000
   cost_scaled: mean=0.0000  std=1.0000


In [112]:
numeric_features = df_positive.select_dtypes(include=np.number).columns.tolist()
numeric_features = [c for c in numeric_features
                    if c != "converted" and not c.endswith("_scaled")]
corr_with_target = (
    df_positive[numeric_features + ["converted"]]
    .corr()["converted"]
    .drop("converted")
    .abs()
    .sort_values(ascending=False)
)
print("\n5. Top 10 features correlated with 'converted':")
print(corr_with_target.head(10).round(4))


5. Top 10 features correlated with 'converted':
revenue                     0.9051
profit                      0.8316
net_margin                  0.5614
position_bias               0.1883
responded                   0.1652
expected_profit             0.0988
response_time_bucket_ord    0.0918
interaction_number          0.0825
efficiency                  0.0820
lead_difficulty             0.0757
Name: converted, dtype: float64
